**MoA 训练与评分一体版**

在 JupyterLab 中选择 **Run → Run All Cells**，依次完成数据读取、预处理、五折训练和评分。
代码默认读取 `C:\cs4486\lish_moa_annotated.csv`。

运行过程中显示数据维度、训练进度、每个候选的折分数和耗时。
训练结束后自动显示运行汇总和评分结果，包括候选参数表、频率基线、固定 C 候选、最优参数及其五折分数。
评分使用每条样本在验证折得到的折外预测，即 OOF 预测。

默认比较 `C=0.01` 和 `C=0.1`，各训练五折，共 10 组折模型。
概率融合权重的比较复用已生成的预测。

比赛按所有“样本 × 标签”的二元 Log Loss 取平均，越低越好。

$$L=-\frac{1}{NM}\sum_{i=1}^{N}\sum_{j=1}^{M}
[y_{ij}\log p_{ij}+(1-y_{ij})\log(1-p_{ij})]$$

概率裁剪到 $[10^{-15},1-10^{-15}]$。每个标签分别预测阳性概率，计算时保留阳性项和阴性项。[Kaggle 评分说明](https://www.kaggle.com/competitions/lish-moa/overview/evaluation)

| 改动 | 方法与原因 |
|---|---|
| 修正标签划分 | 从原始表头锁定 608 个标签，选前 206 个计分标签。按 `g-`、`c-` 和实验条件明确选择输入，避免独热编码改变列顺序后把标签混进输入。 |
| 修正评分 | 实现上面的二元 Log Loss，并用全部折外预测计算总分，按每个样本等权计分。 |
| 利用对照组信息 | 给药样本用于拟合模型，对照样本的预测设为 0。验证集包含两类样本，评分覆盖全部验证行。 |
| 优化实验条件编码 | 将 24、48、72 小时作为类别特征，使模型能够学习每个时间点的独立效应。 |
| 选择正则化强度 | 在相同五折上比较 `C=0.01` 和 `C=0.1`。较小的 C 施加更强的正则化，实际选择由验证损失决定。 |
| 缓解极端概率 | 比较少量训练折阳性率与模型概率的融合比例，降低置信度过高的错误预测带来的损失。 |

注释版数据的最后 608 列及其中前 206 列的含义来自数据作者说明。[数据结构说明](https://github.com/pablormier/kaggle-lish-moa-annotated)
对照样本的标签为 0 来自比赛数据定义，本地数据也已核对这一点。[Kaggle 数据说明](https://www.kaggle.com/competitions/lish-moa/data)
C 与正则化强度的关系见模型文档。[LogisticRegression 文档](https://scikit-learn.org/stable/modules/generated/sklearn.linear_model.LogisticRegression.html)

In [ ]:
from pathlib import Path
from time import perf_counter

import numpy as np
import pandas as pd
from IPython.display import display
from sklearn.compose import ColumnTransformer
from sklearn.impute import SimpleImputer
from sklearn.linear_model import LogisticRegression
from sklearn.model_selection import KFold
from sklearn.multiclass import OneVsRestClassifier
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import OneHotEncoder, StandardScaler
from tqdm.auto import tqdm

DATA_PATH = Path(r"C:\cs4486\lish_moa_annotated.csv")
N_SPLITS = 5
SEED = 42
N_JOBS = -1
REFERENCE_C = 0.1
C_VALUES = (0.01, REFERENCE_C)
PRIOR_WEIGHTS = (0.0, 0.01, 0.05, 0.1)
EPS = 1e-15

**读取数据并固定特征、标签的含义**

原代码在独热编码后取最后 608 列，导致两个计分标签进入输入，并把两个非计分标签当成目标。
这里先从原始表头保存标签名，再独立处理输入列。只读取实际需要的特征和 206 个标签，减少内存占用。

In [ ]:
if not DATA_PATH.is_file():
    raise FileNotFoundError(f"请把 DATA_PATH 改成数据文件的完整路径：{DATA_PATH}")

columns = pd.read_csv(DATA_PATH, nrows=0).columns.tolist()
all_label_columns = columns[-608:]
target_columns = all_label_columns[:206]
numeric_columns = [c for c in columns if c.startswith(("g-", "c-"))]
categorical_columns = ["cp_time", "cp_dose"]
feature_columns = numeric_columns + categorical_columns
metadata_columns = ["sig_id", "drug_id", "training", "cp_type"]

if len(numeric_columns) != 872 or len(all_label_columns) != 608:
    raise ValueError("数据列数与注释版 MoA 数据不一致，请检查输入文件。")
if set(all_label_columns) & set(feature_columns + metadata_columns):
    raise ValueError("输入列与标签列重叠，请检查表头顺序。")
if (target_columns[0], target_columns[-1]) != (
    "5-alpha_reductase_inhibitor", "wnt_inhibitor"
):
    raise ValueError("计分标签的顺序与当前注释版数据不一致。")

dtype_map = {c: np.float32 for c in numeric_columns + target_columns}
dtype_map.update({"sig_id": "string", "drug_id": "string"})
data = pd.read_csv(
    DATA_PATH,
    usecols=metadata_columns + feature_columns + target_columns,
    dtype=dtype_map,
)
train = data.loc[data["training"].eq(True)].reset_index(drop=True)
del data

X = train[feature_columns]
y = train[target_columns].to_numpy(dtype=np.float64)
is_control = train["cp_type"].eq("ctl_vehicle").to_numpy()

if not train["cp_type"].isin(["ctl_vehicle", "trt_cp"]).all():
    raise ValueError("cp_type 中有未识别的实验类型。")
if not np.isin(y, [0.0, 1.0]).all():
    raise ValueError("训练标签应为 0 或 1，请检查缺失值或列选择。")
if np.any(y[is_control] != 0):
    raise ValueError("对照样本含阳性标签，请先核对数据。")

print(f"训练样本数：{len(train):,}")
print(f"给药样本数：{np.count_nonzero(~is_control):,}")
print(f"对照样本数：{np.count_nonzero(is_control):,}")
print(f"连续特征数：{len(numeric_columns)}，实验条件数：{len(categorical_columns)}")
print(f"计分标签数：{y.shape[1]}，阳性标签比例：{y.mean():.4%}")

**评分与预处理**

缺失值填补、标准化和类别编码都在每一折的训练部分拟合。验证数据使用该训练部分得到的变换参数。
这避免了验证数据参与中位数、均值和方差的估计。[scikit-learn 预处理说明](https://scikit-learn.org/stable/common_pitfalls.html)

`mean_multilabel_log_loss` 接收形状相同的真实标签和预测概率，严格按全部二元预测取平均。

In [ ]:
def mean_multilabel_log_loss(y_true, probabilities):
    y_true = np.asarray(y_true, dtype=np.float64)
    probabilities = np.asarray(probabilities, dtype=np.float64)
    if y_true.shape != probabilities.shape:
        raise ValueError("标签矩阵和概率矩阵的形状必须相同。")
    if not np.isfinite(probabilities).all():
        raise ValueError("预测概率含 NaN 或无穷值。")
    if np.any((probabilities < 0.0) | (probabilities > 1.0)):
        raise ValueError("预测概率应位于 0 到 1 之间。")
    p = np.clip(probabilities, EPS, 1.0 - EPS)
    return float(-(y_true * np.log(p) + (1.0 - y_true) * np.log1p(-p)).mean())


def make_preprocessor():
    return ColumnTransformer(
        transformers=[
            (
                "numeric",
                Pipeline([
                    ("imputer", SimpleImputer(strategy="median")),
                    ("scaler", StandardScaler()),
                ]),
                numeric_columns,
            ),
            (
                "categorical",
                OneHotEncoder(
                    categories=[[24, 48, 72], ["D1", "D2"]],
                    drop="first",
                    handle_unknown="ignore",
                    sparse_output=False,
                ),
                categorical_columns,
            ),
        ],
        remainder="drop",
    )

**五折训练与折外预测**

固定 `KFold(n_splits=5, shuffle=True, random_state=42)`，让两个 C 值在同样的训练和验证样本上比较。
每折只拟合一次预处理，再复用给两个候选模型。默认合计训练 10 组折模型，每组包含 206 个二元分类器。

每个标签的训练折阳性率用 $(n_+ + 0.5)/(n+1)$ 做平滑。遇到训练折内只有单一类别的标签时，使用这个平滑概率作为该标签的预测。
模型对其他标签正常训练。对照组的概率在评分前统一设为 0。

代码保留训练警告。若某个标签在训练折内只有单一类别，OneVsRestClassifier 会提示这一情况，下面的平滑概率处理会覆盖该列。
如果出现迭代未收敛的警告，可根据提示调整 `max_iter` 后再评估。

In [ ]:
training_started = perf_counter()
folds = list(KFold(
    n_splits=N_SPLITS, shuffle=True, random_state=SEED
).split(X))

oof_by_c = {c: np.full(y.shape, np.nan) for c in C_VALUES}
prior_oof = np.full(y.shape, np.nan)
fold_records = []

for fold, (train_idx, val_idx) in enumerate(
    tqdm(folds, desc="Folds", total=N_SPLITS), start=1
):
    fit_idx = train_idx[~is_control[train_idx]]
    if len(fit_idx) == 0:
        raise ValueError("当前训练折没有给药样本。")

    y_fit = y[fit_idx]
    positive_counts = y_fit.sum(axis=0)
    prior = (positive_counts + 0.5) / (len(fit_idx) + 1.0)
    constant_targets = (positive_counts == 0) | (positive_counts == len(fit_idx))

    fold_prior = np.broadcast_to(prior, (len(val_idx), y.shape[1])).copy()
    fold_prior[is_control[val_idx]] = 0.0
    prior_oof[val_idx] = fold_prior
    prior_score = mean_multilabel_log_loss(y[val_idx], fold_prior)

    preprocessor = make_preprocessor()
    X_fit = np.ascontiguousarray(
        preprocessor.fit_transform(X.iloc[fit_idx]), dtype=np.float64
    )
    X_val = np.ascontiguousarray(
        preprocessor.transform(X.iloc[val_idx]), dtype=np.float64
    )

    for c in C_VALUES:
        candidate_started = perf_counter()
        classifier = OneVsRestClassifier(
            LogisticRegression(
                C=c,
                solver="liblinear",
                max_iter=1000,
                class_weight=None,
                random_state=SEED,
            ),
            n_jobs=N_JOBS,
        )
        classifier.fit(X_fit, y_fit)
        probabilities = classifier.predict_proba(X_val)
        probabilities[:, constant_targets] = prior[constant_targets]
        probabilities[is_control[val_idx]] = 0.0

        oof_by_c[c][val_idx] = probabilities
        score = mean_multilabel_log_loss(y[val_idx], probabilities)
        elapsed = perf_counter() - candidate_started
        fold_records.append({
            "fold": fold,
            "C": c,
            "train_samples": len(fit_idx),
            "validation_samples": len(val_idx),
            "fit_predict_score_seconds": elapsed,
            "model_log_loss": score,
            "frequency_log_loss": prior_score,
        })
        tqdm.write(
            f"Fold {fold}/{N_SPLITS} | C={c:g} | Log Loss={score:.6f} "
            f"| 频率基线={prior_score:.6f} | 用时={elapsed:.1f} 秒"
        )

if not np.isfinite(prior_oof).all() or any(
    not np.isfinite(predictions).all() for predictions in oof_by_c.values()
):
    raise RuntimeError("折外预测不完整，请先完成全部折的训练。")

training_seconds = perf_counter() - training_started
fold_results = pd.DataFrame(fold_records)
print("\n训练运行结果")
print(f"已完成 {len(fold_records)} 组折模型，每组 {y.shape[1]} 个标签。")
print(f"每个 C 的折外预测矩阵形状：{y.shape}")
print(f"五折训练、预测及折评分总耗时：{training_seconds / 60:.2f} 分钟")
display(fold_results.round(6))

**按比赛标准输出评分结果**

以下单元格在训练完成后自动接着运行，并使用上一步的 OOF 预测进行评分。
融合公式为 $p_{final}=(1-w)p_{model}+w p_{prior}$。
每个样本的先验概率只来自对应训练折的标签统计。
根据所有 OOF 样本的平均二元 Log Loss 选择 C 和融合权重。

输出候选比较表、频率基线、固定 C 候选分数、最优参数，以及最优候选每一折的分数。
保持当前内核时，可以单独重新运行这个单元格查看评分。

In [ ]:
print("评分结果（五折 OOF，越低越好）")
frequency_score = mean_multilabel_log_loss(y, prior_oof)
candidates = []

for c, model_oof in oof_by_c.items():
    for weight in PRIOR_WEIGHTS:
        blended = (1.0 - weight) * model_oof + weight * prior_oof
        per_fold_scores = [
            mean_multilabel_log_loss(y[val_idx], blended[val_idx])
            for _, val_idx in folds
        ]
        candidates.append({
            "C": c,
            "prior_weight": weight,
            "oof_log_loss": mean_multilabel_log_loss(y, blended),
            "fold_mean": np.mean(per_fold_scores),
            "fold_std": np.std(per_fold_scores),
        })

results = pd.DataFrame(candidates).sort_values(
    ["oof_log_loss", "C", "prior_weight"]
).reset_index(drop=True)
best = results.iloc[0]
best_c = float(best["C"])
best_weight = float(best["prior_weight"])
best_oof = (1.0 - best_weight) * oof_by_c[best_c] + best_weight * prior_oof

display(results.round(6))
print(f"同划分的频率基线 OOF Log Loss：{frequency_score:.6f}")
fixed_reference_score = mean_multilabel_log_loss(y, oof_by_c[REFERENCE_C])
print(f"固定 C={REFERENCE_C:g}、融合权重=0 的 OOF Log Loss：{fixed_reference_score:.6f}")
print(f"最佳 C：{best_c:g}，最佳融合权重：{best_weight:g}")
print(f"最佳 OOF Log Loss：{best['oof_log_loss']:.6f}")
print(f"五折均分：{best['fold_mean']:.6f} ± {best['fold_std']:.6f}")
improvement = (frequency_score - best["oof_log_loss"]) / frequency_score
print(f"相对频率基线的损失降低比例：{improvement:.2%}（正数表示降低）")
print(f"相对固定 C={REFERENCE_C:g} 候选的损失降低量：{fixed_reference_score - best['oof_log_loss']:.6f}")

best_fold_results = pd.DataFrame([
    {
        "fold": fold,
        "validation_samples": len(val_idx),
        "best_log_loss": mean_multilabel_log_loss(y[val_idx], best_oof[val_idx]),
        "frequency_log_loss": mean_multilabel_log_loss(y[val_idx], prior_oof[val_idx]),
    }
    for fold, (_, val_idx) in enumerate(folds, start=1)
])
print("\n最优候选的五折评分")
display(best_fold_results.round(6))
print("训练与评分全部完成。")

**如何解读运行结果**

比较表里的 `oof_log_loss` 越低越好。`fold_std` 描述五折之间的波动。
原来的 `2.51720` 使用了不同的评分计算和错位标签，性能比较应从修正后的结果开始。
本版的频率基线同时使用对照组规则与平滑阳性率，因此应以本次计算出的频率基线为参照。
固定 `C=0.1、融合权重=0` 的候选采用本版的特征、对照组和稀有标签处理。
它与最优候选之间的差值反映本次正则化和概率融合调参的收益。

参数和融合权重是按这些 OOF 分数选择的，最优分数包含调参带来的选择偏差。实际性能提升需要通过运行结果确认，独立测试集可进一步评估泛化。
此处的随机五折可能让同一药物的不同实验样本出现在训练和验证两侧，分数应解释为这套随机划分下的表现。评估全新药物时，应另外按 `drug_id` 分组划分。

这份 Notebook 交付时未执行完整五折训练。打开后选择 Run → Run All Cells，即可依次得到训练运行结果和评分结果。

本地评分使用有真实标签的五折验证样本。Kaggle 隐藏测试集由平台评分。运行结果显示在 Notebook 内，保留内核期间可以复用预测重新评分。